# Day 2 - Lab 2: Visualisation with Matplotlib and Seaborn

**Goal:** turn the numbers from Lab 1 into pictures a manager can read in five seconds. Matplotlib for control, Seaborn for speed. End with one chart you would put in front of a director.

A picture is not decoration. The right chart *is* the analysis: it shows the shape a table hides.

## 1. Load

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

def find_data(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / 'data').is_dir():
            return p / 'data'
    raise FileNotFoundError('data/ folder not found')

DATA = find_data()

def _rebuild_clean():
    """Re-run the Day 1 cleaning from raw, so Day 2 is self-contained."""
    df = pd.read_csv(DATA / 'raw' / 'service_requests_raw.csv', dtype=str).drop_duplicates()
    df['priority'] = df['priority'].str.strip().str.title().replace({'2': 'Medium'})
    df['resolution_hours'] = pd.to_numeric(df['resolution_hours'], errors='coerce')
    iso   = pd.to_datetime(df['submitted_at'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    named = pd.to_datetime(df['submitted_at'], format='%d-%b-%Y %H:%M', errors='coerce')
    df['submitted_at'] = iso.fillna(named)
    df['resolved_at'] = pd.to_datetime(df['resolved_at'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    df.loc[(df['resolution_hours'] < 0) | (df['resolution_hours'] > 8760), 'resolution_hours'] = np.nan
    df.loc[(df['resolved_at'] < df['submitted_at']).fillna(False), 'resolved_at'] = pd.NaT
    df['citizen_age_band'] = df['citizen_age_band'].replace({'': pd.NA, 'Unknown': pd.NA}).fillna('Unknown')
    df['service_id'] = pd.to_numeric(df['service_id'], errors='coerce')
    svc = pd.read_csv(DATA / 'seeds' / 'services.csv')
    df = df.merge(svc[['service_id', 'target_resolution_hours']], on='service_id', how='left')
    resolved = df['status'].isin(['Resolved', 'Reopened']) & df['resolution_hours'].notna()
    df['sla_met'] = pd.NA
    df.loc[resolved, 'sla_met'] = (df.loc[resolved, 'resolution_hours'] <= df.loc[resolved, 'target_resolution_hours']).astype('int')
    return df

def load_clean():
    p = DATA / 'processed' / 'service_requests_clean.csv'
    if p.exists():
        print('Loaded cleaned dataset from Day 1:', p.name)
        return pd.read_csv(p, parse_dates=['submitted_at', 'resolved_at'])
    print('Cleaned file not found, rebuilding from raw (Day 1 cleaning)...')
    return _rebuild_clean()

df = load_clean()
channels = pd.read_csv(DATA / 'seeds' / 'channels.csv')
districts = pd.read_csv(DATA / 'seeds' / 'districts.csv')
df = df.merge(channels, on='channel_id', how='left')
print('shape:', df.shape)
df.head(3)

In [ ]:
# TODO: your code here

## 2. Matplotlib fundamentals: figure and axes
Every Matplotlib chart is a **figure** (the canvas) holding one or more **axes** (the plot). Create them explicitly with `plt.subplots()`; it is the habit that scales to multi-panel figures.

In [ ]:
# TODO: your code here

The long tail you measured as skew in Lab 1 is now obvious: most requests cluster low, a few stretch far right.

## 3. Seaborn: the same idea, less code
Seaborn sits on top of Matplotlib and knows about DataFrames. A distribution with a smooth density curve is one line.

In [ ]:
# TODO: your code here

## 4. Compare groups: box and violin plots
A box plot shows median, IQR and outliers per group. Order the boxes by median so the story reads left to right.

In [ ]:
# TODO: your code here

In [ ]:
# A violin plot adds the shape of each distribution
# TODO: your code here

## 5. Categorical counts

In [ ]:
# TODO: your code here

## 6. Correlation heatmap
A heatmap reads a correlation matrix at a glance. Strong red or blue off the diagonal is what you scan for.

In [ ]:
# TODO: your code here

Resolution time tracks the target (0.75), as expected. Satisfaction moves with neither (about 0.00), a real finding: on this data it is not explained by speed.

## 7. A trend over time
Volume by month tells operations whether demand is growing.

In [ ]:
# TODO: your code here

## 8. One chart for the director
Now compose deliberately: a single, annotated comparison that makes the digital-channel finding undeniable. Colour, title and annotation all pull one way.

In [ ]:
# TODO: your code here

## Your turn
1. Draw a box plot of `resolution_hours` by `priority`. Does the spread differ as much as the centre?
2. Plot SLA-met rate by channel as a bar chart (hint: `groupby(...)['sla_met'].mean()`).
3. Take any chart above and improve it for a non-technical audience: title that states the finding, clear axis labels, no clutter.

In [ ]:
# your turn